In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image

# ============================================
# STYLING CONSTANTS
# ============================================
PINK          = "#C11C84"
NODE_BLACK    = "#141D2B"
HACKER_GREY   = "#A4B1CD"
WHITE         = "#FFFFFF"
HTB_GREEN     = "#2E8B57"
MALWARE_RED   = "#FF6B6B"
AZURE         = "#45B7D1"
NUGGET_YELLOW = "#F7DC6F"

# ============================================
# CLASS NAMES
# ============================================
CLASS_NAMES = ['adware', 'backdoor', 'benign', 'downloader',
               'spyware', 'trojan', 'virus', 'worm']

SOURCE_IDX = 7   # worm  (samples to attack)
BENIGN_IDX = 2   # benign (attack target class)

# ============================================
# CONFIGURATION  ← update these paths
# ============================================
DATA_PATH    = r"C:\Users\lenovo LOQ\PFA2\Sorted"
MODEL_A_PATH = "model_a_vgg16.pth"  # VGG16  — attacked model
MODEL_B_PATH = "malware_classifier.pth"                                  # ResNet50 — transfer target
OUTPUT_DIR   = "pgd_targeted_output_vgg16"
NUM_EXAMPLES = 30

# PGD hyper-parameters
EPSILON = 8 / 255
ALPHA   = 2 / 255
N_ITER  = 40

# ── output folders ────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "adversarial_images"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "original_images"),   exist_ok=True)


# ============================================
# MODEL A — VGG16  (attacked model)
# ============================================
class ModelA_VGG16(nn.Module):
    """VGG16-based malware classifier — same architecture as DeepFool notebook."""
    def __init__(self, num_classes: int = 8):
        super().__init__()
        self.model = models.vgg16(weights=None)
        self.model.classifier[6] = nn.Linear(
            self.model.classifier[6].in_features, num_classes
        )

    def forward(self, x):
        return self.model(x)


def load_model_A(path: str, device: torch.device) -> nn.Module:
    """Load VGG16 checkpoint (expects dict with 'model_state_dict')."""
    model = ModelA_VGG16(num_classes=8)
    ckpt  = torch.load(path, map_location=device)
    state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state)
    model.to(device).eval()
    val_acc = ckpt.get('val_acc', 'N/A') if isinstance(ckpt, dict) else 'N/A'
    epoch   = ckpt.get('epoch',   'N/A') if isinstance(ckpt, dict) else 'N/A'
    print(f"[✓] Loaded Model A (VGG16) — epoch {epoch}, val_acc {val_acc}")
    return model


# ============================================
# MODEL B — ResNet50  (transfer target)
# ============================================
class ModelB_ResNet50(nn.Module):
    """ResNet50-based malware classifier — same architecture as targeted_pgd notebook."""
    def __init__(self, n_classes: int = 8):
        super().__init__()
        self.resnet = models.resnet50(weights='DEFAULT')
        for param in self.resnet.parameters():
            param.requires_grad = False
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Linear(num_features, 1000),
            nn.ReLU(),
            nn.Linear(1000, n_classes)
        )

    def forward(self, x):
        return self.resnet(x)


def load_model_B(path: str, device: torch.device) -> nn.Module:
    """Load ResNet50 — tries TorchScript first, falls back to state-dict."""
    try:
        model = torch.jit.load(path, map_location=device)
        print("[✓] Loaded Model B (ResNet50 — TorchScript)")
    except Exception:
        model = ModelB_ResNet50(n_classes=8)
        model.load_state_dict(torch.load(path, map_location=device))
        print("[✓] Loaded Model B (ResNet50 — state-dict)")
    model.to(device).eval()
    return model


# ============================================
# DATA
# ============================================
def load_data(data_path: str) -> datasets.ImageFolder:
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std =[0.229, 0.224, 0.225])
    ])
    dataset = datasets.ImageFolder(os.path.join(data_path, "test"), transform=transform)
    print(f"[✓] Test dataset loaded: {len(dataset)} samples")
    return dataset


# ============================================
# TARGETED PGD
# ============================================
def pgd_targeted(
    model       : nn.Module,
    x           : torch.Tensor,   # (1, C, H, W) on device
    target_class: int,
    epsilon     : float = EPSILON,
    alpha       : float = ALPHA,
    n_iter      : int   = N_ITER,
    random_start: bool  = True,
) -> torch.Tensor:
    """
    L-∞ targeted PGD.
    Minimises CE of target_class to push the model toward predicting benign.
    Returns the adversarial image (same shape as x).
    """
    criterion = nn.CrossEntropyLoss()
    target    = torch.tensor([target_class], device=x.device)
    x_adv     = x.clone().detach()

    if random_start:
        noise = torch.empty_like(x_adv).uniform_(-epsilon, epsilon)
        x_adv = (x_adv + noise).detach()

    for _ in range(n_iter):
        x_adv.requires_grad_(True)
        loss = criterion(model(x_adv), target)
        model.zero_grad()
        loss.backward()

        with torch.no_grad():
            # Gradient descent to minimise target-class loss
            x_adv = x_adv - alpha * x_adv.grad.sign()
            # Project back into ε-ball around original x
            delta = torch.clamp(x_adv - x, min=-epsilon, max=epsilon)
            x_adv = (x + delta).detach()

    return x_adv


# ============================================
# HELPERS
# ============================================
def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean


def save_image(tensor: torch.Tensor, path: str) -> None:
    img = denormalize(tensor.cpu()).squeeze()
    img = torch.clamp(img, 0, 1).permute(1, 2, 0).numpy()
    img = (img * 255).astype(np.uint8)
    Image.fromarray(img).save(path)


def _ax_defaults(axes_flat):
    for ax in axes_flat:
        ax.set_facecolor(NODE_BLACK)
        for spine in ax.spines.values():
            spine.set_edgecolor(HACKER_GREY)
        ax.tick_params(colors=WHITE)


# ============================================
# PHASE 1 — ATTACK VGG16 (white-box)
# ============================================
def generate_pgd_attacks(model_A, dataset, device, num_examples):
    """
    Run targeted PGD (worm → benign) on Model A (VGG16).
    Only attacks worm samples that VGG16 correctly classifies.
    """
    print(f"\n{'='*60}")
    print(f"TARGETED PGD: worm → benign  |  Model A (VGG16)")
    print(f"ε={EPSILON*255:.0f}/255  α={ALPHA*255:.0f}/255  iters={N_ITER}")
    print(f"{'='*60}\n")

    worm_indices = [i for i, (_, lbl) in enumerate(dataset.samples)
                    if lbl == SOURCE_IDX]

    results       = []
    success_count = 0
    processed     = 0

    for idx in worm_indices:
        if processed >= num_examples:
            break

        x, true_lbl = dataset[idx]
        x = x.unsqueeze(0).to(device)

        # Only attack correctly-classified worms
        with torch.no_grad():
            probs_clean = F.softmax(model_A(x), dim=1)
            pred_clean  = probs_clean.argmax(1).item()
            conf_before = probs_clean[0, SOURCE_IDX].item()

        if pred_clean != SOURCE_IDX:
            continue

        # ── PGD attack on VGG16 ──────────────────────────────────────────────
        x_adv = pgd_targeted(model_A, x, BENIGN_IDX)

        with torch.no_grad():
            probs_adv  = F.softmax(model_A(x_adv), dim=1)
            pred_adv   = probs_adv.argmax(1).item()
            conf_after = probs_adv[0, pred_adv].item()

        success = (pred_adv == BENIGN_IDX)
        if success:
            success_count += 1

        perturbation = (x_adv - x).detach()
        linf_norm    = torch.abs(perturbation.cpu()).max().item()

        orig_path = os.path.join(OUTPUT_DIR, "original_images",
            f"sample_{processed:03d}_worm.png")
        adv_path  = os.path.join(OUTPUT_DIR, "adversarial_images",
            f"sample_{processed:03d}_worm_to_{CLASS_NAMES[pred_adv]}.png")

        save_image(x.detach(),     orig_path)
        save_image(x_adv.detach(), adv_path)

        results.append({
            'idx'               : processed,
            'original_image'    : x.detach().cpu(),
            'adversarial_image' : x_adv.detach().cpu(),
            'perturbation'      : perturbation.detach().cpu(),
            'true_label'        : SOURCE_IDX,
            'original_label'    : SOURCE_IDX,
            'adversarial_label_A': pred_adv,
            'original_conf_A'   : conf_before,
            'adversarial_conf_A': conf_after,
            'success_A'         : success,
            'linf_norm'         : linf_norm,
            'orig_img_path'     : orig_path,
            'adv_img_path'      : adv_path,
        })

        status = "✓" if success else "✗"
        print(f"  {status} Sample {processed+1:3d}: worm → "
              f"{CLASS_NAMES[pred_adv]:10s} | "
              f"L∞: {linf_norm:6.4f} | "
              f"Conf: {conf_before:.3f}→{conf_after:.3f}")

        processed += 1

    print(f"\n{'='*60}")
    print(f"ATTACK SUMMARY  (Model A — VGG16)")
    print(f"{'='*60}")
    print(f"  Success rate : {success_count}/{processed} "
          f"({100*success_count/max(processed,1):.1f}%)")
    print(f"  Avg L∞       : {np.mean([r['linf_norm'] for r in results]):.4f}  "
          f"(ε = {EPSILON:.4f})")
    print(f"{'='*60}\n")
    return results


# ============================================
# PHASE 2 — TRANSFERABILITY TO ResNet50
# ============================================
def evaluate_transferability(results, model_B, device):
    """
    Feed adversarial worm images (that fooled VGG16) into ResNet50
    and measure how many it also classifies as benign.
    Mirrors the DeepFool transferability evaluation exactly.
    """
    print(f"\n{'='*60}")
    print("TRANSFERABILITY EVALUATION  (Model A → Model B)")
    print("Attack   : Targeted PGD  worm → benign")
    print("Victim   : Model A (VGG16)")
    print("Transfer : Model B (ResNet50)")
    print(f"{'='*60}")

    # Only test images that actually fooled VGG16
    successful_A = [r for r in results if r['success_A']]

    transfer_to_benign = 0
    transfer_any       = 0
    confs_B            = []
    per_sample_details = []

    for r in successful_A:
        adv = r['adversarial_image'].to(device)

        with torch.no_grad():
            out   = model_B(adv)
            probs = F.softmax(out, dim=1)
            pred  = out.argmax(1).item()
            conf  = probs[0, pred].item()

        confs_B.append(conf)
        fooled_B = (pred != SOURCE_IDX)   # any non-worm = general transfer
        benign_B = (pred == BENIGN_IDX)   # specifically benign = targeted transfer

        if fooled_B:
            transfer_any += 1
        if benign_B:
            transfer_to_benign += 1

        per_sample_details.append({
            **r,
            'adversarial_label_B' : pred,
            'adversarial_conf_B'  : conf,
            'transfer_any'        : fooled_B,
            'transfer_benign'     : benign_B,
        })

    total_A = len(successful_A)
    if total_A == 0:
        print("[!] No successful adversarial examples to transfer.")
        return []

    print(f"\n  Fooled Model A (VGG16) worm→benign : {total_A}/{len(results)}")
    print(f"  Transfer — classified benign (ResNet50) : {transfer_to_benign}/{total_A}  "
          f"({100*transfer_to_benign/total_A:.1f}%)")
    print(f"  Transfer — any misclassification        : {transfer_any}/{total_A}  "
          f"({100*transfer_any/total_A:.1f}%)")
    print(f"  Avg confidence of ResNet50              : {np.mean(confs_B):.4f}")

    # ── per-sample breakdown ─────────────────────────────────────────────────
    print(f"\n  {'#':>4}  {'VGG16 pred':>12}  {'ResNet50 pred':>14}  "
          f"{'ResNet50 conf':>14}  {'Transfer?':>10}")
    print("  " + "-"*60)
    for d in per_sample_details:
        tag = "benign ✓" if d['transfer_benign'] else \
              ("any ✓"   if d['transfer_any']    else "✗ worm")
        print(f"  {d['idx']:>4}  "
              f"{CLASS_NAMES[d['adversarial_label_A']]:>12}  "
              f"{CLASS_NAMES[d['adversarial_label_B']]:>14}  "
              f"{d['adversarial_conf_B']:>13.3f}  "
              f"{tag:>10}")

    print(f"{'='*60}\n")
    return per_sample_details


# ============================================
# SUMMARY
# ============================================
def print_summary(results, details):
    print(f"\n{'='*60}")
    print("FULL ATTACK + TRANSFER SUMMARY")
    print(f"{'='*60}")

    succ_A = [r for r in results if r['success_A']]
    print(f"\n  [Model A — VGG16]  Targeted PGD  worm → benign")
    print(f"    Success (worm→benign) : {len(succ_A)}/{len(results)}  "
          f"({100*len(succ_A)/max(len(results),1):.1f}%)")
    print(f"    Avg L∞               : {np.mean([r['linf_norm'] for r in results]):.4f}  "
          f"(ε = {EPSILON:.4f})")

    if details:
        transfer_benign = sum(1 for d in details if d['transfer_benign'])
        transfer_any    = sum(1 for d in details if d['transfer_any'])
        print(f"\n  [Model B — ResNet50]  Transferability")
        print(f"    Inputs                     : {len(details)} adversarials that fooled VGG16")
        print(f"    → classified as benign     : {transfer_benign}/{len(details)}  "
              f"({100*transfer_benign/len(details):.1f}%)")
        print(f"    → any misclassification    : {transfer_any}/{len(details)}  "
              f"({100*transfer_any/len(details):.1f}%)")
        print(f"    Avg ResNet50 confidence    : {np.mean([d['adversarial_conf_B'] for d in details]):.4f}")

    print(f"{'='*60}\n")


# ============================================
# VISUALIZATION: ATTACK GRID
# ============================================
def visualize_attack_grid(results, save_dir, n_samples=10):
    """3-column grid: original worm | perturbation heatmap | adversarial (VGG16)."""
    print("[i] Generating attack grid …")
    n = min(n_samples, len(results))
    fig, axes = plt.subplots(n, 3, figsize=(15, 5*n))
    fig.patch.set_facecolor(NODE_BLACK)
    if n == 1:
        axes = axes.reshape(1, -1)
    _ax_defaults(axes.flatten())

    for i, r in enumerate(results[:n]):
        orig = np.clip(denormalize(r['original_image'].squeeze())
                       .permute(1,2,0).numpy(), 0, 1)
        adv  = np.clip(denormalize(r['adversarial_image'].squeeze())
                       .permute(1,2,0).numpy(), 0, 1)
        pert = r['perturbation'].squeeze().numpy()
        mag  = np.sqrt(np.sum(pert**2, axis=0))

        axes[i,0].imshow(orig)
        axes[i,0].set_title(f"Original: worm\nConf: {r['original_conf_A']:.3f}",
                             color=HTB_GREEN, fontsize=10, fontweight='bold')
        axes[i,0].axis('off')

        axes[i,1].imshow(mag, cmap='hot')
        axes[i,1].set_title(f"Perturbation\nL∞: {r['linf_norm']:.4f}",
                             color=NUGGET_YELLOW, fontsize=10, fontweight='bold')
        axes[i,1].axis('off')

        tc = MALWARE_RED if r['success_A'] else HACKER_GREY
        axes[i,2].imshow(adv)
        axes[i,2].set_title(
            f"Adversarial (VGG16): {CLASS_NAMES[r['adversarial_label_A']]}\n"
            f"Conf: {r['adversarial_conf_A']:.3f}",
            color=tc, fontsize=10, fontweight='bold')
        axes[i,2].axis('off')

    plt.suptitle('Targeted PGD (worm → benign) — VGG16 Attack Grid',
                 color=HTB_GREEN, fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'attack_grid.png'),
                facecolor=NODE_BLACK, dpi=200, bbox_inches='tight')
    plt.close()
    print("[✓] attack_grid.png saved")


# ============================================
# VISUALIZATION: TRANSFER GRID
# ============================================
def visualize_transfer_grid(details, save_dir, n_samples=10):
    """
    3-column grid:
    original worm | adversarial (VGG16 says benign) | ResNet50 verdict
    """
    print("[i] Generating transferability grid …")
    n = min(n_samples, len(details))
    fig, axes = plt.subplots(n, 3, figsize=(15, 5*n))
    fig.patch.set_facecolor(NODE_BLACK)
    if n == 1:
        axes = axes.reshape(1, -1)
    _ax_defaults(axes.flatten())

    for i, d in enumerate(details[:n]):
        orig = np.clip(denormalize(d['original_image'].squeeze())
                       .permute(1,2,0).numpy(), 0, 1)
        adv  = np.clip(denormalize(d['adversarial_image'].squeeze())
                       .permute(1,2,0).numpy(), 0, 1)

        axes[i,0].imshow(orig)
        axes[i,0].set_title("Original\n(worm)",
                             color=HTB_GREEN, fontsize=10, fontweight='bold')
        axes[i,0].axis('off')

        axes[i,1].imshow(adv)
        axes[i,1].set_title(
            f"Adversarial\nVGG16: {CLASS_NAMES[d['adversarial_label_A']]}  "
            f"conf={d['adversarial_conf_A']:.3f}",
            color=MALWARE_RED, fontsize=10, fontweight='bold')
        axes[i,1].axis('off')

        tc = MALWARE_RED  if d['transfer_benign'] else \
             NUGGET_YELLOW if d['transfer_any']    else HACKER_GREY
        lbl = CLASS_NAMES[d['adversarial_label_B']]
        tag = "(benign ✓)" if d['transfer_benign'] else \
              "(misclf ✓)" if d['transfer_any']    else "(worm  ✗)"
        axes[i,2].imshow(adv)
        axes[i,2].set_title(
            f"ResNet50: {lbl} {tag}\nConf: {d['adversarial_conf_B']:.3f}",
            color=tc, fontsize=10, fontweight='bold')
        axes[i,2].axis('off')

    plt.suptitle('Transferability: VGG16 PGD adversarials → ResNet50',
                 color=HTB_GREEN, fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'transfer_grid.png'),
                facecolor=NODE_BLACK, dpi=200, bbox_inches='tight')
    plt.close()
    print("[✓] transfer_grid.png saved")


# ============================================
# VISUALIZATION: METRICS
# ============================================
def visualize_metrics(results, details, save_dir):
    """
    4-panel dashboard:
    confidence drop (VGG16) | ResNet50 prediction breakdown |
    confidence comparison   | success summary bar
    """
    print("[i] Generating metrics dashboard …")
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.patch.set_facecolor(NODE_BLACK)
    _ax_defaults(axes.flatten())
    for ax in axes.flatten():
        ax.grid(True, alpha=0.3, color=HACKER_GREY, linestyle='--')

    # ── Panel 1: VGG16 confidence drop (before → after attack) ───────────────
    conf_before = [r['original_conf_A']    for r in results]
    conf_after  = [r['adversarial_conf_A'] for r in results]
    x_pos = np.arange(len(results))
    w = 0.38
    axes[0,0].bar(x_pos - w/2, conf_before, w, label='Before attack (worm conf)',
                  color=HTB_GREEN, alpha=0.75, edgecolor=HACKER_GREY)
    axes[0,0].bar(x_pos + w/2, conf_after,  w, label='After attack (pred conf)',
                  color=MALWARE_RED, alpha=0.75, edgecolor=HACKER_GREY)
    axes[0,0].set_xlabel('Sample index', color=WHITE, fontsize=12)
    axes[0,0].set_ylabel('Confidence',   color=WHITE, fontsize=12)
    axes[0,0].set_title('VGG16 Confidence: Before vs After Attack',
                        color=HTB_GREEN, fontsize=14, fontweight='bold')
    axes[0,0].legend(facecolor=NODE_BLACK, edgecolor=HACKER_GREY, labelcolor=WHITE)
    axes[0,0].set_ylim(0, 1.1)

    # ── Panel 2: ResNet50 prediction breakdown ────────────────────────────────
    if details:
        vgg_preds = [CLASS_NAMES[d['adversarial_label_B']] for d in details]
        unique, counts = np.unique(vgg_preds, return_counts=True)
        colors_bar = [HTB_GREEN    if c == 'benign' else
                      MALWARE_RED  if c == 'worm'   else AZURE
                      for c in unique]
        bars = axes[0,1].bar(unique, counts, color=colors_bar,
                             alpha=0.8, edgecolor=HACKER_GREY, linewidth=1.2)
        for bar, cnt in zip(bars, counts):
            axes[0,1].text(bar.get_x() + bar.get_width()/2,
                           bar.get_height() + 0.1, str(cnt),
                           ha='center', color=WHITE, fontsize=11, fontweight='bold')
        axes[0,1].set_xlabel('ResNet50 Prediction', color=WHITE, fontsize=12)
        axes[0,1].set_ylabel('Count',               color=WHITE, fontsize=12)
        axes[0,1].set_title('ResNet50 Prediction Distribution\n(on transferred adversarials)',
                            color=HTB_GREEN, fontsize=14, fontweight='bold')
        axes[0,1].tick_params(axis='x', rotation=30)

    # ── Panel 3: Confidence comparison VGG16 vs ResNet50 ─────────────────────
    if details:
        c_vgg    = [d['adversarial_conf_A'] for d in details]
        c_resnet = [d['adversarial_conf_B'] for d in details]
        x2 = np.arange(len(details))
        axes[1,0].bar(x2 - w/2, c_vgg,    w, label='VGG16 conf',    color=HTB_GREEN,
                      alpha=0.75, edgecolor=HACKER_GREY)
        axes[1,0].bar(x2 + w/2, c_resnet,  w, label='ResNet50 conf', color=PINK,
                      alpha=0.75, edgecolor=HACKER_GREY)
        axes[1,0].set_xlabel('Sample index', color=WHITE, fontsize=12)
        axes[1,0].set_ylabel('Confidence',   color=WHITE, fontsize=12)
        axes[1,0].set_title('Model Confidence on Adversarial Images\n(VGG16 vs ResNet50)',
                            color=HTB_GREEN, fontsize=14, fontweight='bold')
        axes[1,0].legend(facecolor=NODE_BLACK, edgecolor=HACKER_GREY, labelcolor=WHITE)
        axes[1,0].set_ylim(0, 1.1)

    # ── Panel 4: Transfer summary bar ─────────────────────────────────────────
    if details:
        total         = len(details)
        benign_count  = sum(1 for d in details if d['transfer_benign'])
        any_count     = sum(1 for d in details if d['transfer_any'] and not d['transfer_benign'])
        worm_count    = total - sum(1 for d in details if d['transfer_any'])

        categories = ['→ benign\n(targeted)', '→ other\n(any transfer)', '→ worm\n(no transfer)']
        values     = [benign_count, any_count, worm_count]
        colors     = [MALWARE_RED, NUGGET_YELLOW, HACKER_GREY]

        bars = axes[1,1].bar(categories, values, color=colors,
                             alpha=0.8, edgecolor=HACKER_GREY, linewidth=1.2)
        for bar, val in zip(bars, values):
            pct = 100 * val / total if total > 0 else 0
            axes[1,1].text(bar.get_x() + bar.get_width()/2,
                           bar.get_height() + 0.1,
                           f'{val}\n({pct:.1f}%)',
                           ha='center', color=WHITE, fontsize=11, fontweight='bold')
        axes[1,1].set_ylabel('Count', color=WHITE, fontsize=12)
        axes[1,1].set_title('Transferability Breakdown\n(ResNet50 outcomes)',
                            color=HTB_GREEN, fontsize=14, fontweight='bold')
        axes[1,1].set_ylim(0, total + 3)

    plt.suptitle('Targeted PGD — VGG16 Attack & ResNet50 Transferability Metrics',
                 color=HTB_GREEN, fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'attack_metrics.png'),
                facecolor=NODE_BLACK, dpi=200, bbox_inches='tight')
    plt.close()
    print("[✓] attack_metrics.png saved")


# ============================================
# VISUALIZATION: SINGLE DETAILED
# ============================================
def visualize_single_detailed(result, detail, save_dir):
    """
    4-panel deep-dive for one sample:
    original | perturbation magnitude | adversarial (VGG16) | ResNet50 verdict
    """
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.patch.set_facecolor(NODE_BLACK)
    _ax_defaults(axes)

    orig = np.clip(denormalize(result['original_image'].squeeze())
                   .permute(1,2,0).numpy(), 0, 1)
    adv  = np.clip(denormalize(result['adversarial_image'].squeeze())
                   .permute(1,2,0).numpy(), 0, 1)
    pert = result['perturbation'].squeeze().numpy()
    mag  = np.sqrt(np.sum(pert**2, axis=0))

    axes[0].imshow(orig)
    axes[0].set_title(f"Original\nworm  conf={result['original_conf_A']:.3f}",
                      color=HTB_GREEN, fontweight='bold')
    axes[0].axis('off')

    im = axes[1].imshow(mag, cmap='viridis')
    axes[1].set_title(f"Perturbation magnitude\nL∞={result['linf_norm']:.4f}  (ε={EPSILON:.4f})",
                      color=NUGGET_YELLOW, fontweight='bold')
    axes[1].axis('off')
    cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    cbar.ax.tick_params(colors=WHITE)

    axes[2].imshow(adv)
    axes[2].set_title(
        f"VGG16 (victim)\n→ {CLASS_NAMES[result['adversarial_label_A']]}  "
        f"conf={result['adversarial_conf_A']:.3f}",
        color=MALWARE_RED, fontweight='bold')
    axes[2].axis('off')

    if detail:
        tc  = MALWARE_RED  if detail['transfer_benign'] else \
              NUGGET_YELLOW if detail['transfer_any']    else HACKER_GREY
        lbl = CLASS_NAMES[detail['adversarial_label_B']]
        tag = "benign ✓" if detail['transfer_benign'] else \
              "misclf ✓" if detail['transfer_any']    else "worm ✗"
        axes[3].imshow(adv)
        axes[3].set_title(
            f"ResNet50 (transfer)\n→ {lbl} [{tag}]  conf={detail['adversarial_conf_B']:.3f}",
            color=tc, fontweight='bold')
    else:
        axes[3].set_visible(False)
    axes[3].axis('off')

    fig.text(0.5, 0.01,
             f"Targeted PGD  ε={EPSILON*255:.0f}/255  α={ALPHA*255:.0f}/255  iters={N_ITER}  |  "
             f"Attack: VGG16 (white-box)  →  Transfer: ResNet50 (black-box)",
             ha='center', color=WHITE, fontsize=10, fontweight='bold')
    plt.suptitle("Targeted PGD — Detailed Single-Sample Analysis",
                 color=HTB_GREEN, fontsize=15, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'detailed_sample_{result["idx"]:03d}.png'),
                facecolor=NODE_BLACK, dpi=300, bbox_inches='tight')
    plt.close()


# ============================================
# MAIN
# ============================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[i] Device: {device}\n")

    # ── Load models ──────────────────────────────────────────────────────────
    print("[i] Loading Model A (VGG16 — victim) …")
    model_A = load_model_A(MODEL_A_PATH, device)

    print("\n[i] Loading Model B (ResNet50 — transfer target) …")
    model_B = load_model_B(MODEL_B_PATH, device)

    # ── Load data ─────────────────────────────────────────────────────────────
    print("\n[i] Loading test dataset …")
    dataset = load_data(DATA_PATH)

    # ── Phase 1: Targeted PGD on VGG16 (white-box) ───────────────────────────
    results = generate_pgd_attacks(model_A, dataset, device, NUM_EXAMPLES)

    # ── Phase 2: Transferability to ResNet50 (black-box) ─────────────────────
    details = evaluate_transferability(results, model_B, device)

    # ── Summary ───────────────────────────────────────────────────────────────
    print_summary(results, details)

    # ── Visualizations ────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("GENERATING VISUALIZATIONS")
    print(f"{'='*60}\n")

    visualize_attack_grid(results, OUTPUT_DIR, n_samples=10)
    visualize_transfer_grid(details, OUTPUT_DIR, n_samples=10)
    visualize_metrics(results, details, OUTPUT_DIR)

    if results:
        first_detail = details[0] if details else None
        visualize_single_detailed(results[0], first_detail, OUTPUT_DIR)
        print("[✓] detailed_sample_000.png saved")

    # ── Save results ──────────────────────────────────────────────────────────
    torch.save({'results': results, 'transfer_details': details},
               os.path.join(OUTPUT_DIR, "pgd_transfer_results.pt"))
    print("[✓] pgd_transfer_results.pt saved")

    print(f"\n{'='*60}")
    print("✓ ALL DONE")
    print(f"{'='*60}")
    print(f"\nOutputs in '{OUTPUT_DIR}/':")
    print("  attack_grid.png          — original | perturbation | VGG16 adversarial")
    print("  transfer_grid.png        — VGG16 adversarial vs ResNet50 verdict")
    print("  attack_metrics.png       — confidence drops + ResNet50 breakdown")
    print("  detailed_sample_000.png  — 4-panel deep-dive for sample 0")
    print(f"  adversarial_images/      — {len(results)} adversarial PNGs")
    print(f"  original_images/         — {len(results)} original PNGs")
    print(f"  pgd_transfer_results.pt  — all result dicts\n")


if __name__ == "__main__":
    main()

[i] Device: cuda

[i] Loading Model A (VGG16 — victim) …


C:\Users\lenovo LOQ\AppData\Local\Temp\ipykernel_11992\994214244.py:72: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt  = torch.load(path, map_location=device)


[✓] Loaded Model A (VGG16) — epoch 39, val_acc 97.98277425203989

[i] Loading Model B (ResNet50 — transfer target) …
[✓] Loaded Model B (ResNet50 — TorchScript)

[i] Loading test dataset …
[✓] Test dataset loaded: 2211 samples

TARGETED PGD: worm → benign  |  Model A (VGG16)
ε=8/255  α=2/255  iters=40

  ✓ Sample   1: worm → benign     | L∞: 0.0314 | Conf: 1.000→1.000
  ✓ Sample   2: worm → benign     | L∞: 0.0314 | Conf: 1.000→1.000
  ✓ Sample   3: worm → benign     | L∞: 0.0314 | Conf: 1.000→1.000
  ✓ Sample   4: worm → benign     | L∞: 0.0314 | Conf: 1.000→1.000
  ✓ Sample   5: worm → benign     | L∞: 0.0314 | Conf: 0.977→1.000
  ✓ Sample   6: worm → benign     | L∞: 0.0314 | Conf: 1.000→1.000
  ✓ Sample   7: worm → benign     | L∞: 0.0314 | Conf: 1.000→1.000
  ✓ Sample   8: worm → benign     | L∞: 0.0314 | Conf: 1.000→1.000
  ✓ Sample   9: worm → benign     | L∞: 0.0314 | Conf: 1.000→1.000
  ✓ Sample  10: worm → benign     | L∞: 0.0314 | Conf: 1.000→1.000
  ✓ Sample  11: worm → ben